# RedisCachingModel - Automatic Model-Level Caching

This notebook demonstrates `RedisCachingModel`, a transparent caching wrapper for the OpenAI Agents SDK `Model` interface.

## How It Works

`RedisCachingModel` wraps any SDK `Model` implementation and intercepts `get_response()` calls to check a 2-level cache before forwarding to the underlying model:

- **Level 1 (L1) - Exact Match**: A SHA256 hash of `(system_instructions, input)` is used as a Redis key. If the exact same prompt was seen before, the cached response is returned immediately.
- **Level 2 (L2) - Semantic Similarity**: If L1 misses and semantic caching is enabled, the query is compared against previous queries using vector embeddings. If a sufficiently similar query is found (above the configurable threshold), its cached response is returned.

For `stream_response()`, caching is bypassed entirely since streaming delivers responses incrementally.

## Cache Bypass Rules

Caching is automatically bypassed when any of the following are present in the request:
- **Tools** - Responses may depend on dynamic tool call results
- **Handoffs** - Complex agent interactions are not cacheable
- **Output schema** - Structured output validation needs fresh responses

## Benefits

- Reduces LLM API calls by ~25%
- Reduces latency by ~30% for cached responses
- Completely transparent to the agent and runner

In [ ]:
import os
import asyncio
import json
import time
from dataclasses import dataclass
from typing import Any
from collections.abc import AsyncIterator

from redis_openai_agents import RedisCachingModel, CachedModelResponse, CachedUsage

# Configuration
REDIS_URL = os.getenv("REDIS_URL", "redis://localhost:6379")

print(f"Redis URL: {REDIS_URL}")

## 1. Understanding the SDK Model Interface

`RedisCachingModel` implements the same interface as the OpenAI Agents SDK `Model`. The SDK defines two key methods on any model:

- `get_response(system_instructions, input, model_settings, tools, output_schema, handoffs, tracing, ...)` - Returns a complete response
- `stream_response(...)` - Returns an async iterator of streaming events

`RedisCachingModel` wraps an existing model and intercepts these calls:
- `get_response()` checks the cache first, and only calls the underlying model on a cache miss
- `stream_response()` always delegates directly to the underlying model (no caching for streams)

To demonstrate without making real API calls, we will create a mock model that simulates the SDK `Model` interface.

In [ ]:
@dataclass
class MockUsage:
    """Simulates the Usage object from the SDK."""
    input_tokens: int = 100
    output_tokens: int = 50
    requests: int = 1


@dataclass
class MockOutputItem:
    """Simulates an output item with model_dump support."""
    role: str = "assistant"
    content: str = ""

    def model_dump(self, exclude_unset: bool = False) -> dict:
        return {"role": self.role, "content": self.content}


@dataclass
class MockModelResponse:
    """Simulates the ModelResponse from the SDK."""
    output: list
    usage: MockUsage
    response_id: str = "resp_mock_001"

    def to_input_items(self) -> list:
        return [item.model_dump() for item in self.output]


class MockModel:
    """A mock model that simulates the SDK Model interface.

    This tracks call count so we can verify when caching prevents
    calls to the underlying model.
    """

    def __init__(self):
        self.call_count = 0
        self.responses = {}

    def set_response(self, input_text: str, response_text: str):
        """Pre-configure a response for a given input."""
        self.responses[input_text] = response_text

    async def get_response(
        self,
        system_instructions=None,
        input=None,
        model_settings=None,
        tools=None,
        output_schema=None,
        handoffs=None,
        tracing=None,
        *,
        previous_response_id=None,
        conversation_id=None,
        prompt=None,
    ):
        self.call_count += 1
        response_text = self.responses.get(
            input, f"Mock response for: {input}"
        )
        print(f"  [MockModel] Call #{self.call_count}: generating response for '{input}'")
        await asyncio.sleep(0.05)  # Simulate API latency
        return MockModelResponse(
            output=[MockOutputItem(content=response_text)],
            usage=MockUsage(),
            response_id=f"resp_mock_{self.call_count:03d}",
        )

    def stream_response(self, *args, **kwargs):
        raise NotImplementedError("Streaming not implemented in mock")


print("Mock model classes defined.")

## 2. Creating a RedisCachingModel Wrapper

The `RedisCachingModel` constructor accepts:

| Parameter | Type | Default | Description |
|-----------|------|---------|-------------|
| `model` | `Model` | (required) | The underlying model to wrap |
| `redis_url` | `str` | `redis://localhost:6379` | Redis connection URL |
| `cache_prefix` | `str` | `model_cache` | Prefix for cache keys in Redis |
| `cache_ttl` | `int` | `3600` | Time-to-live for cache entries (seconds) |
| `enable_semantic_cache` | `bool` | `False` | Enable L2 semantic similarity caching |
| `semantic_threshold` | `float` | `0.95` | Minimum similarity for semantic cache hit |

After construction, you must call `await initialize()` to establish the Redis connection.

In [ ]:
# Create the mock model
mock_model = MockModel()
mock_model.set_response("What is Redis?", "Redis is an in-memory data store.")
mock_model.set_response("What is Python?", "Python is a programming language.")
mock_model.set_response(
    "Explain caching strategies",
    "Common caching strategies include write-through, write-back, and cache-aside.",
)

# Wrap with RedisCachingModel
cached_model = RedisCachingModel(
    model=mock_model,
    redis_url=REDIS_URL,
    cache_prefix="demo_caching_model",
    cache_ttl=300,  # 5 minutes for this demo
)

# Initialize Redis connection
await cached_model.initialize()

print("RedisCachingModel initialized.")
print(f"  Underlying model: {type(mock_model).__name__}")
print(f"  Cache prefix: demo_caching_model")
print(f"  Cache TTL: 300s")
print(f"  Semantic cache: disabled")

## 3. Exact Cache Hits (L1)

The L1 cache uses an exact match strategy. A SHA256 hash of `(system_instructions, input)` serves as the cache key. When the same combination is seen again, the cached response is returned without calling the underlying model.

Let's call `get_response()` twice with identical inputs and observe that the second call is served from cache.

In [ ]:
# Reset call count to track model invocations
mock_model.call_count = 0

# First call - cache miss, will call the underlying model
print("--- First call: 'What is Redis?' ---")
t1 = time.time()
response1 = await cached_model.get_response(
    system_instructions="You are a helpful assistant.",
    input="What is Redis?",
    model_settings=None,
    tools=[],
    output_schema=None,
    handoffs=[],
    tracing=None,
)
elapsed1 = time.time() - t1
print(f"  Response: {response1.output}")
print(f"  Time: {elapsed1:.4f}s")
print(f"  Model calls so far: {mock_model.call_count}")

print()

# Second call - exact same input, should be an L1 cache hit
print("--- Second call: 'What is Redis?' (identical) ---")
t2 = time.time()
response2 = await cached_model.get_response(
    system_instructions="You are a helpful assistant.",
    input="What is Redis?",
    model_settings=None,
    tools=[],
    output_schema=None,
    handoffs=[],
    tracing=None,
)
elapsed2 = time.time() - t2
print(f"  Response: {response2.output}")
print(f"  Time: {elapsed2:.4f}s")
print(f"  Model calls so far: {mock_model.call_count}")

print()
print(f"Result: Model was called {mock_model.call_count} time(s) for 2 requests.")
print(f"  The second call was served from L1 cache ({elapsed2:.4f}s vs {elapsed1:.4f}s).")

In [ ]:
# Different system instructions produce a different cache key,
# even with the same input text
print("--- Same input, different system instructions ---")
response3 = await cached_model.get_response(
    system_instructions="You are a database expert.",
    input="What is Redis?",
    model_settings=None,
    tools=[],
    output_schema=None,
    handoffs=[],
    tracing=None,
)
print(f"  Model calls so far: {mock_model.call_count}")
print(f"  A new model call was made because the system instructions changed.")

## 4. Semantic Cache Hits (L2)

When `enable_semantic_cache=True`, the wrapper also checks for semantically similar queries using vector embeddings. This catches cases where the user asks the same question with different wording.

The L2 cache uses the `SemanticCache` from `redis_openai_agents.cache`, which stores vector embeddings of queries and finds matches above the `semantic_threshold` (default: 0.95).

**Note**: Semantic caching requires the `sentence-transformers` package and will download the embedding model on first use.

In [ ]:
# Create a new cached model with semantic caching enabled
mock_model_semantic = MockModel()
mock_model_semantic.set_response(
    "What is Redis?",
    "Redis is an in-memory data structure store.",
)

cached_model_semantic = RedisCachingModel(
    model=mock_model_semantic,
    redis_url=REDIS_URL,
    cache_prefix="demo_semantic_cache",
    cache_ttl=300,
    enable_semantic_cache=True,   # Enable L2
    semantic_threshold=0.90,      # 90% similarity threshold
)

await cached_model_semantic.initialize()

print("RedisCachingModel with semantic cache initialized.")
print(f"  Semantic cache: enabled")
print(f"  Similarity threshold: 0.90")

In [ ]:
# First call - populates both L1 exact cache and L2 semantic cache
print("--- Call 1: 'What is Redis?' ---")
response_s1 = await cached_model_semantic.get_response(
    system_instructions="You are a helpful assistant.",
    input="What is Redis?",
    model_settings=None,
    tools=[],
    output_schema=None,
    handoffs=[],
    tracing=None,
)
print(f"  Model calls: {mock_model_semantic.call_count}")

# Second call - semantically similar but not exact
# This would miss L1 (different hash) but could hit L2 (similar meaning)
print("\n--- Call 2: 'Can you tell me about Redis?' (semantically similar) ---")
response_s2 = await cached_model_semantic.get_response(
    system_instructions="You are a helpful assistant.",
    input="Can you tell me about Redis?",
    model_settings=None,
    tools=[],
    output_schema=None,
    handoffs=[],
    tracing=None,
)
print(f"  Model calls: {mock_model_semantic.call_count}")

# Check metrics to see if semantic hit was recorded
metrics = await cached_model_semantic.get_metrics()
print(f"\nMetrics: {json.dumps(metrics, indent=2)}")
print(f"  Semantic hits: {metrics['semantic_hits']}")

## 5. Cache Bypass (When Tools Are Present)

The `_should_bypass_cache()` method returns `True` when any of the following are non-empty:
- `tools` - Tool-augmented responses depend on dynamic tool call results
- `handoffs` - Multi-agent handoff interactions are not cacheable
- `output_schema` - Structured output validation requires fresh responses

When cache is bypassed, the request goes directly to the underlying model and the response is **not** stored in cache.

In [ ]:
# Reset for clean demonstration
mock_model.call_count = 0

# Simulate a tool definition
mock_tool = {"name": "web_search", "description": "Search the web"}

# Call with tools present - cache is bypassed
print("--- Call with tools (cache bypassed) ---")
response_tools1 = await cached_model.get_response(
    system_instructions="You are a helpful assistant.",
    input="What is Redis?",
    model_settings=None,
    tools=[mock_tool],       # Tools present -> bypass cache
    output_schema=None,
    handoffs=[],
    tracing=None,
)
print(f"  Model calls: {mock_model.call_count}")

# Same call again with tools - still bypasses cache
print("\n--- Same call with tools again (still bypassed) ---")
response_tools2 = await cached_model.get_response(
    system_instructions="You are a helpful assistant.",
    input="What is Redis?",
    model_settings=None,
    tools=[mock_tool],       # Tools present -> bypass cache
    output_schema=None,
    handoffs=[],
    tracing=None,
)
print(f"  Model calls: {mock_model.call_count}")

print(f"\nBoth calls went to the model because tools were present.")
print(f"The model was called {mock_model.call_count} times for 2 requests.")

In [ ]:
# Demonstrate bypass with handoffs
print("--- Call with handoffs (cache bypassed) ---")
mock_handoff = {"agent": "specialist", "description": "Hand off to specialist"}
response_handoff = await cached_model.get_response(
    system_instructions="You are a helpful assistant.",
    input="What is Redis?",
    model_settings=None,
    tools=[],
    output_schema=None,
    handoffs=[mock_handoff],  # Handoffs present -> bypass cache
    tracing=None,
)
print(f"  Model calls: {mock_model.call_count} (bypassed due to handoffs)")

# Demonstrate bypass with output_schema
print("\n--- Call with output_schema (cache bypassed) ---")
mock_schema = {"type": "object", "properties": {"answer": {"type": "string"}}}
response_schema = await cached_model.get_response(
    system_instructions="You are a helpful assistant.",
    input="What is Redis?",
    model_settings=None,
    tools=[],
    output_schema=mock_schema,  # Schema present -> bypass cache
    handoffs=[],
    tracing=None,
)
print(f"  Model calls: {mock_model.call_count} (bypassed due to output_schema)")

## 6. Cache Metrics (`get_metrics()`)

The `get_metrics()` method returns a dictionary with cache performance statistics:

| Metric | Description |
|--------|-------------|
| `cache_hits` | Number of L1 exact match hits |
| `cache_misses` | Number of cache misses (including bypasses) |
| `semantic_hits` | Number of L2 semantic similarity hits |
| `hit_rate` | Ratio of `hits / (hits + misses)` |

These metrics are tracked in-memory via the `CacheMetrics` dataclass and reset when the `RedisCachingModel` is re-created.

In [ ]:
# Create a fresh cached model for clean metrics
mock_model_metrics = MockModel()
cached_model_metrics = RedisCachingModel(
    model=mock_model_metrics,
    redis_url=REDIS_URL,
    cache_prefix="demo_metrics",
    cache_ttl=300,
)
await cached_model_metrics.initialize()

# Make a series of calls
queries = [
    "What is Redis?",           # Miss (first time)
    "What is Python?",          # Miss (first time)
    "What is Redis?",           # Hit (exact match)
    "What is Python?",          # Hit (exact match)
    "What is Redis?",           # Hit (exact match)
    "Explain caching",          # Miss (new query)
]

print("Executing queries:")
print("=" * 50)
for i, query in enumerate(queries, 1):
    calls_before = mock_model_metrics.call_count
    await cached_model_metrics.get_response(
        system_instructions=None,
        input=query,
        model_settings=None,
        tools=[],
        output_schema=None,
        handoffs=[],
        tracing=None,
    )
    was_hit = mock_model_metrics.call_count == calls_before
    status = "HIT" if was_hit else "MISS"
    print(f"  Query {i}: '{query}' -> {status}")

# Get metrics
print("\nCache Metrics:")
print("=" * 50)
metrics = await cached_model_metrics.get_metrics()
print(f"  Cache hits:    {metrics['cache_hits']}")
print(f"  Cache misses:  {metrics['cache_misses']}")
print(f"  Semantic hits: {metrics['semantic_hits']}")
print(f"  Hit rate:      {metrics['hit_rate']:.1%}")
print(f"  Model calls:   {mock_model_metrics.call_count} (out of {len(queries)} requests)")

## 7. TTL Configuration

The `cache_ttl` parameter controls how long cached responses are stored in Redis (in seconds). After the TTL expires, the cached entry is automatically removed by Redis, and the next identical request will result in a cache miss.

Choosing the right TTL depends on your use case:
- **Short TTL (60-300s)**: Good for rapidly changing data or when freshness matters
- **Medium TTL (3600s)**: Default; good balance for most agent workloads
- **Long TTL (86400s+)**: Good for stable, reference-style queries

In [ ]:
# Demonstrate TTL behavior with a very short TTL
mock_model_ttl = MockModel()

cached_model_ttl = RedisCachingModel(
    model=mock_model_ttl,
    redis_url=REDIS_URL,
    cache_prefix="demo_ttl",
    cache_ttl=2,  # 2 seconds - very short for demonstration
)
await cached_model_ttl.initialize()

print("Created RedisCachingModel with cache_ttl=2 seconds")

# First call - cache miss
print("\n--- Call 1: cache miss ---")
await cached_model_ttl.get_response(
    system_instructions=None,
    input="Short-lived query",
    model_settings=None,
    tools=[],
    output_schema=None,
    handoffs=[],
    tracing=None,
)
print(f"  Model calls: {mock_model_ttl.call_count}")

# Immediate second call - cache hit
print("\n--- Call 2 (immediate): cache hit ---")
await cached_model_ttl.get_response(
    system_instructions=None,
    input="Short-lived query",
    model_settings=None,
    tools=[],
    output_schema=None,
    handoffs=[],
    tracing=None,
)
print(f"  Model calls: {mock_model_ttl.call_count}")

# Wait for TTL to expire
print("\n--- Waiting 3 seconds for TTL to expire... ---")
await asyncio.sleep(3)

# Third call after TTL expiry - cache miss again
print("\n--- Call 3 (after TTL expiry): cache miss ---")
await cached_model_ttl.get_response(
    system_instructions=None,
    input="Short-lived query",
    model_settings=None,
    tools=[],
    output_schema=None,
    handoffs=[],
    tracing=None,
)
print(f"  Model calls: {mock_model_ttl.call_count}")

print(f"\nThe model was called {mock_model_ttl.call_count} times for 3 requests.")
print("Call 2 was a cache hit. Call 3 was a miss because the TTL expired.")

await cached_model_ttl.close()

## 8. Integration with Agent

In a real application, you would wrap a production model (like `OpenAIResponsesModel`) with `RedisCachingModel` and pass it directly to an `Agent` or `Runner`. The caching is completely transparent.

```python
from agents import Agent, Runner
from agents.models import OpenAIResponsesModel
from redis_openai_agents import RedisCachingModel

# 1. Create the base model
base_model = OpenAIResponsesModel(model="gpt-4o")

# 2. Wrap with caching
cached_model = RedisCachingModel(
    model=base_model,
    redis_url="redis://localhost:6379",
    cache_ttl=3600,              # 1 hour
    enable_semantic_cache=True,   # Enable L2 semantic matching
    semantic_threshold=0.95,      # High similarity required
)
await cached_model.initialize()

# 3. Use with Agent - the agent has no idea caching is happening
agent = Agent(
    name="CachedAssistant",
    instructions="You are a helpful assistant.",
    model=cached_model,  # <-- pass the caching wrapper as the model
)

# 4. Run the agent
result = await Runner.run(agent, "What is Redis?")
print(result.final_output)

# 5. Same question again - served from cache, no API call
result2 = await Runner.run(agent, "What is Redis?")
print(result2.final_output)

# 6. Check metrics
metrics = await cached_model.get_metrics()
print(f"Hit rate: {metrics['hit_rate']:.1%}")

# 7. Cleanup
await cached_model.close()
```

**Key points:**
- The `Agent` constructor accepts any object that implements the SDK `Model` interface
- `RedisCachingModel` implements `get_response()` and `stream_response()`, matching the interface
- When tools are attached to the agent, caching is automatically bypassed for those calls
- `stream_response()` always delegates directly to the underlying model (no caching)

In [ ]:
# Demonstrate the pattern with our mock model
# (This simulates what happens internally when Runner.run calls model.get_response)

mock_agent_model = MockModel()
mock_agent_model.set_response(
    "Tell me about vector databases",
    "Vector databases store and search high-dimensional vectors for similarity search.",
)

agent_cached_model = RedisCachingModel(
    model=mock_agent_model,
    redis_url=REDIS_URL,
    cache_prefix="demo_agent_integration",
    cache_ttl=600,
)
await agent_cached_model.initialize()

# Simulate what Runner.run does internally:
# It calls model.get_response() which our wrapper intercepts
print("Simulating Runner.run() -> model.get_response():")
print("=" * 55)

for i in range(3):
    calls_before = mock_agent_model.call_count
    response = await agent_cached_model.get_response(
        system_instructions="You are a helpful assistant.",
        input="Tell me about vector databases",
        model_settings=None,
        tools=[],
        output_schema=None,
        handoffs=[],
        tracing=None,
    )
    was_cached = mock_agent_model.call_count == calls_before
    source = "CACHE" if was_cached else "MODEL"
    print(f"  Run {i+1}: source={source}, output={response.output}")

metrics = await agent_cached_model.get_metrics()
print(f"\nFinal metrics: {json.dumps(metrics, indent=2)}")
print(f"API calls saved: {len(range(3)) - mock_agent_model.call_count}")

await agent_cached_model.close()

## Summary

`RedisCachingModel` provides transparent, 2-level caching for the OpenAI Agents SDK `Model` interface:

| Feature | Description |
|---------|-------------|
| **L1 Exact Cache** | SHA256 hash of `(system_instructions, input)` for instant exact-match lookups |
| **L2 Semantic Cache** | Vector similarity search for catching rephrased queries |
| **Automatic Bypass** | Cache is skipped when tools, handoffs, or output schemas are present |
| **Stream Passthrough** | `stream_response()` always delegates to the underlying model |
| **TTL Expiration** | Configurable time-to-live ensures cache freshness |
| **Metrics** | Built-in hit/miss/semantic hit tracking via `get_metrics()` |
| **Drop-in Replacement** | Pass as `model=` to any `Agent` or `Runner.run()` |

### Architecture

```
Agent / Runner
    |
    v
RedisCachingModel.get_response()
    |
    |-- tools/handoffs/schema present? --> bypass --> underlying Model
    |
    |-- L1 exact match (Redis Hash)? --> return cached
    |
    |-- L2 semantic match (RedisVL)? --> return cached
    |
    |-- cache miss --> call underlying Model --> store in cache --> return
```

In [ ]:
# Cleanup: close all remaining connections
await cached_model.close()
await cached_model_metrics.close()

# Close semantic model if it was initialized
try:
    await cached_model_semantic.close()
except Exception:
    pass

print("All connections closed.")